# 01 - Exploração e Tratamento dos Dados

Este notebook apresenta o processo de exploração, tratamento e
preparação dos dados utilizados no projeto ARCAG.

O projeto utiliza o dataset **cit-HepTh**, disponibilizado pelo
**Stanford Network Analysis Project (SNAP)**. O dataset representa uma
rede direcionada de citações entre artigos científicos da área de
Física de Altas Energias.

Nesta etapa, os dados brutos serão analisados e transformados em
estruturas adequadas para:

- armazenamento das informações dos artigos;
- representação dos relacionamentos de citação;
- construção e análise dos grafos;
- desenvolvimento dos mecanismos de busca;
- cálculo de métricas relacionadas à relevância dos artigos.

O processo será dividido em quatro etapas principais:

1. **Exploração:** compreensão da estrutura e das características dos
   dados disponíveis;
2. **Tratamento:** limpeza, normalização e padronização das informações;
3. **Integração:** combinação das diferentes fontes de dados;
4. **Preparação:** geração dos dados processados que serão utilizados
   posteriormente pelo backend e pela aplicação.

## 1. Dataset

O dataset utilizado neste projeto é o **cit-HepTh**, uma rede de
citações de artigos da categoria High Energy Physics - Theory (HEP-TH)
do arXiv.

O conjunto de dados disponibilizado pelo SNAP contém informações sobre
os relacionamentos de citação entre os artigos e informações
complementares sobre os trabalhos científicos.

Para este projeto, serão utilizadas três fontes principais:

| Arquivo | Informação |
|---|---|
| `cit-HepTh.txt.gz` | Relacionamentos de citação entre artigos |
| `cit-HepTh-dates.txt.gz` | Data de submissão dos artigos |
| `cit-HepTh-abstracts.tar.gz` | Metadados e resumo dos artigos |

A rede de citações é direcionada: quando um artigo `A` cita um artigo
`B`, é estabelecida uma relação `A → B`.

Essas relações serão utilizadas posteriormente para a construção da
rede de citações e para o cálculo de métricas relacionadas aos artigos.

## 2. Configuração do ambiente

Nesta etapa são definidos os diretórios utilizados pelo notebook.

Os dados originais são mantidos em `data/raw`, enquanto os resultados
do processo de tratamento serão armazenados em `data/processed`.

O dataset original não será alterado durante o processo de tratamento.

In [ ]:
import sys
import pandas as pd
from pathlib import Path

SRC_DIR = Path.cwd().parent

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_DATA_DIR = SRC_DIR / "data" / "raw"
PROCESSED_DATA_DIR = SRC_DIR / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Diretório fonte: {SRC_DIR}")
print(f"Dados brutos: {RAW_DATA_DIR}")
print(f"Dados processados: {PROCESSED_DATA_DIR}")

In [ ]:
list(RAW_DATA_DIR.iterdir())

## 3. Carregamento dos dados brutos

Os dados serão carregados por meio das funções de processamento
implementadas em `data.preprocess`.

A separação entre o notebook e as funções de processamento permite
que a lógica desenvolvida nesta etapa possa ser reutilizada
posteriormente pelo backend, evitando a duplicação de código.

In [ ]:
from data.preprocess import (
    load_citations,
    load_dates,
    parse_abstract,
    extract_articles,
)

citations = load_citations(
    RAW_DATA_DIR / "cit-HepTh.txt.gz"
)

dates = load_dates(
    RAW_DATA_DIR / "cit-HepTh-dates.txt.gz"
)

## 4. Exploração inicial

Após o carregamento, os dados serão analisados para identificar sua
estrutura, quantidade de registros, tipos de dados e possíveis
inconsistências.

Essa etapa é importante para determinar quais tratamentos serão
necessários antes da integração das diferentes fontes.

In [ ]:
print("Citations:")
display(citations.head())

print(f"Quantidade de relações: {len(citations)}")
print(f"Dimensões: {citations.shape}")

print("Tipos de dados - citations:")
display(citations.dtypes)

## 5. Análise da qualidade dos dados

Antes da integração das fontes, serão verificadas possíveis
inconsistências nos dados, incluindo:

- valores nulos;
- registros duplicados;
- identificadores vazios;
- relações de citação duplicadas;
- formatos inconsistentes de datas.

Os resultados dessa análise serão utilizados para definir as etapas
de tratamento necessárias.

In [ ]:
print("Valores nulos - citations:")
display(citations.isnull().sum())


print(f"Relações duplicadas: {citations.duplicated().sum()}")

## 6. Extração dos metadados dos artigos

O arquivo `cit-HepTh-abstracts.tar.gz` contém os metadados dos artigos
em arquivos individuais organizados por ano.

Cada arquivo possui informações como:

- identificador do artigo;
- título;
- autores;
- data;
- comentários;
- classificação temática;
- referência bibliográfica;
- resumo do artigo.

Um exemplo da estrutura encontrada é:

```text
Paper: hep-th/9201001
From: ...
Date: ...
Title: ...
Authors: ...
Comments: ...
Subj-class: ...
Journal-ref: ...
\\
Abstract...
\\

In [ ]:
import tarfile

abstracts_path = RAW_DATA_DIR / "cit-HepTh-abstracts.tar.gz"

## quantity
with tarfile.open(abstracts_path, "r:gz") as tar:
    members = tar.getmembers()

print(f"Quantidade de entradas no arquivo: {len(members)}")

for member in members[:20]:
    print(member.name)

## example
with tarfile.open(abstracts_path, "r:gz") as tar:
    member = tar.getmember("1992/9201001.abs")

    with tar.extractfile(member) as file:
        content = file.read().decode("utf-8")

print(content)

## 7. Métricas de citação

A partir da rede de citações, serão calculadas métricas básicas para
cada artigo.

Serão consideradas inicialmente duas medidas:

- **Citações recebidas:** quantidade de artigos que citam determinado
  artigo;
- **Citações realizadas:** quantidade de artigos citados por determinado
  artigo.

Na representação da rede direcionada, essas medidas correspondem,
respectivamente, ao grau de entrada (`in-degree`) e ao grau de saída
(`out-degree`) de cada artigo.

In [ ]:
citations_received = (
    citations
    .groupby("cited_paper")
    .size()
    .reset_index(name="citations_received")
)

citations_made = (
    citations
    .groupby("citing_paper")
    .size()
    .reset_index(name="citations_made")
)

In [ ]:
display(citations_received.head())

In [ ]:
display(citations_made.head())

## 8. Associação entre artigos e relacionamentos

Nesta etapa foi estabelecida a correspondência entre os identificadores presentes na rede de citações e os arquivos de metadados dos artigos.

O arquivo `cit-HepTh.txt.gz` utiliza identificadores numéricos que podem possuir quantidade variável de dígitos. Já os arquivos `.abs` utilizam identificadores com sete dígitos.

Para estabelecer a correspondência entre as duas fontes, os identificadores da rede foram normalizados adicionando zeros à esquerda até atingir sete dígitos.

Por exemplo:

| ID na rede | ID normalizado |
| ---------- | -------------- |
| `1001`     | `0001001`      |
| `1002`     | `0001002`      |
| `9304045`  | `9304045`      |

Após a normalização, os 27.770 identificadores únicos presentes na rede de citações possuem um arquivo `.abs` correspondente.

Dessa forma, foi estabelecida a correspondência entre os nós da rede e os metadados dos artigos.


### 8.1 Identificação dos identificadores

Primeiramente, foram obtidos os identificadores únicos presentes na rede de citações e os identificadores disponíveis nos arquivos `.abs`.

In [ ]:
import tarfile
from pathlib import Path

citation_ids = set(citations["citing_paper"]) | set(citations["cited_paper"])

with tarfile.open(abstracts_path, "r:gz") as tar:
    abstract_ids = {
        Path(member.name).stem
        for member in tar.getmembers()
        if member.name.endswith(".abs")
    }

### 8.2 Normalização e validação dos identificadores

Foi criada uma função para normalizar os identificadores da rede, adicionando zeros à esquerda até completar sete caracteres.

In [ ]:
def normalize_citation_id(paper_id: str) -> str:
    return paper_id.zfill(7)

normalized_citation_ids = {
    normalize_citation_id(paper_id)
    for paper_id in citation_ids
}

common_ids = normalized_citation_ids & abstract_ids

print(f"IDs únicos na rede: {len(citation_ids)}")
print(f"IDs normalizados encontrados nos abstracts: {len(common_ids)}")

missing_ids = normalized_citation_ids - abstract_ids

print(f"IDs sem abstract correspondente: {len(missing_ids)}")

### 8.3 Extração dos metadados

Os arquivos `.abs` contêm as informações individuais de cada artigo. Antes da extração de todos os arquivos, foi utilizado um arquivo como teste para validar o processo de leitura e interpretação dos metadados.

In [ ]:
with tarfile.open(abstracts_path, "r:gz") as tar:
    member = tar.getmember("1992/9201001.abs")
    file = tar.extractfile(member)

    content = file.read().decode("utf-8", errors="replace")

article = parse_abstract(content)

article

In [ ]:
articles = extract_articles(abstracts_path)

print(f"Quantidade de artigos extraídos: {len(articles)}")

### 8.4 Validação dos metadados extraídos

Após a extração, foram realizadas verificações para avaliar a estrutura e a qualidade dos metadados obtidos.

In [ ]:
articles.info()

In [ ]:
articles.isnull().sum()

Os valores ausentes encontrados em campos como autores, classificação temática, comentários e referência bibliográfica correspondem a informações que não estão disponíveis em determinados registros da fonte original.

Os campos essenciais para a identificação e utilização dos artigos, como `paper_id`, `title` e `abstract`, estão preenchidos.

Também foi verificada a existência de identificadores duplicados.

In [ ]:
print(f"Total de artigos: {len(articles)}")
print(f"IDs únicos: {articles['paper_id'].nunique()}")
print(f"Artigos duplicados: {articles['paper_id'].duplicated().sum()}")

O resultado confirmou que os 29.555 artigos extraídos possuem identificadores únicos.

In [ ]:
articles.head()

### 8.5 Normalização dos identificadores dos artigos

Os metadados extraídos utilizam o identificador no formato `hep-th/YYMMNNN`, enquanto a rede de citações utiliza apenas a parte numérica do identificador.

Para permitir a associação entre as duas fontes, foi criada uma coluna `network_id`, contendo a representação numérica normalizada utilizada pela rede.

In [ ]:
articles["network_id"] = (
    articles["paper_id"]
    .str.split("/")
    .str[-1]
    .str.zfill(7)
)

In [ ]:
articles[["network_id", "paper_id", "title"]].head()

In [ ]:
article_network_ids = set(articles["network_id"])

common_ids = article_network_ids & normalized_citation_ids
missing_ids = normalized_citation_ids - article_network_ids

print(f"IDs na rede: {len(normalized_citation_ids)}")
print(f"IDs encontrados nos artigos: {len(common_ids)}")
print(f"IDs da rede sem artigo: {len(missing_ids)}")

## 9. Integração dos dados

Após a validação dos identificadores, as informações dos artigos foram integradas às métricas derivadas da rede de citações.

Para isso, os identificadores utilizados nas tabelas de citações foram normalizados utilizando a mesma representação `network_id` adotada na tabela de artigos.

In [ ]:
citations_received["network_id"] = (
    citations_received["cited_paper"]
    .map(normalize_citation_id)
)

citations_made["network_id"] = (
    citations_made["citing_paper"]
    .map(normalize_citation_id)
)

Em seguida, as métricas de citações recebidas e realizadas foram associadas aos respectivos artigos por meio da coluna `network_id`.

In [ ]:
papers = articles.merge(
    citations_received[["network_id", "citations_received"]],
    on="network_id",
    how="left"
)

papers = papers.merge(
    citations_made[["network_id", "citations_made"]],
    on="network_id",
    how="left"
)

A tabela resultante reúne os metadados dos artigos e as métricas derivadas da rede de citações.

In [ ]:
papers.head()

In [ ]:
print(f"Total de artigos: {len(papers)}")
print(f"Colunas: {list(papers.columns)}")

A tabela `papers` passa a representar a principal estrutura de dados dos artigos utilizada pelo projeto.

## 10. Validação dos dados processados

Após a integração das diferentes fontes, foram realizadas verificações para garantir a consistência da tabela final.

Inicialmente, foram identificados os artigos que não possuíam registros nas tabelas de citações.


In [ ]:
papers[["citations_received", "citations_made"]].isnull().sum()

A ausência de registro nessas métricas não representa necessariamente um erro. Um artigo que não aparece como `cited_paper` não recebeu citações dentro da rede, enquanto um artigo que não aparece como `citing_paper` não realizou citações.

Dessa forma, esses valores foram convertidos para zero.

In [ ]:
papers["citations_received"] = (
    papers["citations_received"]
    .fillna(0)
    .astype(int)
)

papers["citations_made"] = (
    papers["citations_made"]
    .fillna(0)
    .astype(int)
)

A ausência de valores nulos nas métricas foi confirmada:

In [ ]:
papers[["citations_received", "citations_made"]].isnull().sum()

Também foi realizada uma inspeção da estrutura final:

In [ ]:
papers[
    [
        "network_id",
        "paper_id",
        "title",
        "citations_received",
        "citations_made"
    ]
].head(10)

A quantidade de artigos e a unicidade dos identificadores foram verificadas:

In [ ]:
print(f"Artigos na tabela final: {len(papers)}")
print(f"IDs únicos: {papers['network_id'].nunique()}")

Também foi analisada a presença de valores ausentes nas demais colunas:

In [ ]:
papers.isnull().sum()

Por fim, foi verificada a consistência entre a tabela final de artigos e a quantidade de relações da rede.

In [ ]:
print(f"Total de citações realizadas: {papers['citations_made'].sum()}")
print(f"Total de citações recebidas: {papers['citations_received'].sum()}")

## 11. Exportação dos dados processados

Após o tratamento, integração e validação, os dados foram exportados para arquivos CSV no diretório data/processed.

Foram gerados dois arquivos:

- `papers.csv`: informações dos artigos e métricas de citação;
- `citations.csv`: relacionamentos de citação entre os artigos.

Esses arquivos serão utilizados como base para as próximas etapas do projeto, incluindo a implementação da API, os mecanismos de busca e a construção dos grafos.

In [ ]:
papers.to_csv(
    PROCESSED_DATA_DIR / "papers.csv",
    index=False
)

citations.to_csv(
    PROCESSED_DATA_DIR / "citations.csv",
    index=False
)

Os arquivos gerados podem ser verificados no diretório de dados processados:

In [ ]:
print("Arquivos processados:")
print(PROCESSED_DATA_DIR / "papers.csv")
print(PROCESSED_DATA_DIR / "citations.csv")